In [40]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
from sklearn.linear_model import LinearRegression
import datetime
import folium
from folium.plugins import HeatMap

In [51]:
!jupyter trust "C:/Users/user/Developer/projeiett/TahminHarita.ipynb"

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: kernel kernelspec migrate run troubleshoot

Jupyter command `jupyter-trust` not found.


In [42]:
#yolculuk
yolculuk_chunksize = 500_000
yolculuk_summary_list = []

for chunk in pd.read_csv(r"C:\Users\user\Developer\projeiett\ocak_ayi.csv", 
                        chunksize=yolculuk_chunksize, parse_dates=['transition_date']):
    chunk['date'] = chunk['transition_date'].dt.date
    grouped = chunk.groupby(['date','town'], as_index=False)['number_of_passenger'].sum()
    yolculuk_summary_list.append(grouped)

daily_passenger = pd.concat(yolculuk_summary_list, ignore_index=True)
daily_passenger = daily_passenger.groupby(['date','town'], as_index=False)['number_of_passenger'].sum()
display(daily_passenger.head())


,date,town,number_of_passenger
0,2023-01-01,ADALAR,14597
1,2023-01-01,ATASEHIR,59295
2,2023-01-01,AVCILAR,57162
3,2023-01-01,BAGCILAR,55665
4,2023-01-01,BAHCELIEVLER,30134


In [43]:
#trafik
trafik_chunksize = 500_000
trafik_summary_list = []

for chunk in pd.read_csv(r"C:\Users\user\Developer\projeiett\Trafik.csv",
                        chunksize=trafik_chunksize, parse_dates=['trafficindexdate']):
    chunk['date'] = chunk['trafficindexdate'].dt.date
    grouped = chunk.groupby('date', as_index=False)['average_traffic_index'].mean()
    trafik_summary_list.append(grouped)

daily_traffic = pd.concat(trafik_summary_list, ignore_index=True)
daily_traffic = daily_traffic.groupby('date', as_index=False)['average_traffic_index'].mean()
display(daily_traffic.head())

C:\Users\user\AppData\Local\Temp\ipykernel_6028\2594183106.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  for chunk in pd.read_csv(r"C:\Users\user\Developer\projeiett\Trafik.csv",


,date,average_traffic_index
0,2015-08-06,57.858116
1,2015-08-07,23.770492
2,2015-08-11,38.601266
3,2015-08-12,29.715278
4,2015-08-13,28.557491


In [44]:
#hava
hava_chunksize = 500_000
hava_summary_list = []

for chunk in pd.read_csv(r"C:\Users\user\Developer\projeiett\Hava.csv",
                        chunksize=hava_chunksize, parse_dates=['datetime']):
    chunk['date'] = chunk['datetime'].dt.date
    grouped = chunk.groupby('date', as_index=False)[['temp','precip','humidity']].mean()
    hava_summary_list.append(grouped)

daily_weather = pd.concat(hava_summary_list, ignore_index=True)
daily_weather = daily_weather.groupby('date', as_index=False)[['temp','precip','humidity']].mean()
display(daily_weather.head())

,date,temp,precip,humidity
0,2013-01-01,7.8,0.0,71.5
1,2013-01-02,7.0,0.0,81.8
2,2013-01-03,7.9,0.0,85.4
3,2013-01-04,6.7,0.0,83.7
4,2013-01-05,7.0,0.6,73.6


In [45]:
#birleştir,normalize et
merged = pd.merge(daily_passenger, daily_traffic, on='date', how='left')
merged = pd.merge(merged, daily_weather, on='date', how='left')

display(merged.head())


,date,town,number_of_passenger,average_traffic_index,temp,precip,humidity
0,2023-01-01,ADALAR,14597,16.927083,8.4,0.0,92.6
1,2023-01-01,ATASEHIR,59295,16.927083,8.4,0.0,92.6
2,2023-01-01,AVCILAR,57162,16.927083,8.4,0.0,92.6
3,2023-01-01,BAGCILAR,55665,16.927083,8.4,0.0,92.6
4,2023-01-01,BAHCELIEVLER,30134,16.927083,8.4,0.0,92.6


In [46]:
#tahmin 
merged_model = merged.dropna()

X = merged_model[['average_traffic_index','temp','precip','humidity']]
y = merged_model['number_of_passenger']

model = LinearRegression()
model.fit(X, y)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [47]:
#kullanıcı giriş
user_date_str = input("Tahmin için tarihi gir (YYYY-MM-DD): ")
user_town = input("Tahmin için ilçeyi gir: ")

user_date = datetime.datetime.strptime(user_date_str, "%Y-%m-%d").date()
print(f"Tahmin yapılacak tarih: {user_date}, İlçe: {user_town}")


Tahmin yapılacak tarih: 2023-01-20, İlçe: Esenler


In [48]:
row_traffic = daily_traffic[daily_traffic['date'] == user_date]
row_weather = daily_weather[daily_weather['date'] == user_date]

if row_traffic.empty or row_weather.empty:
    print("Bu tarih için veri yok.")
else:
    X_user = pd.DataFrame({
        'average_traffic_index': row_traffic['average_traffic_index'].values,
        'temp': row_weather['temp'].values,
        'precip': row_weather['precip'].values,
        'humidity': row_weather['humidity'].values
    })
    
    prediction = model.predict(X_user)
    print(f"Tahmini yolcu sayısı: {int(prediction[0])} kişi")


Tahmini yolcu sayısı: 186825 kişi


In [49]:
def on_click(event):
    print(f"Tıklanan koordinat: {event['lat']}, {event['lng']}")

m = folium.Map(location=[41.0082, 28.9784], zoom_start=11)
m.add_child(folium.LatLngPopup())  # Haritaya tıklayınca lat/lng gösterir
m
# İstanbul koordinatları (merkez)
istanbul_center = [41.0082, 28.9784]

m = folium.Map(location=istanbul_center, zoom_start=11)

# Örnek: sadece kullanıcı ilçesi için marker
folium.CircleMarker(
    location=[41.0, 28.9],  # Örnek koordinat, gerçek harita için ilçeye göre
    radius=10,
    popup=f"{user_town}: {int(prediction[0])} yolcu",
    color='red',
    fill=True,
    fill_color='red'
).add_to(m)

m


In [50]:
from folium.plugins import HeatMap

# Harita oluştur
m = folium.Map(location=[41.0082, 28.9784], zoom_start=11)

# Heatmap için data: [lat, lon, weight]
# weight → tahmini yolcu sayısı
heat_data = [
    [41.0, 28.9, int(prediction[0])],   # Örnek Esenler
    [40.98, 29.0, 800]                  # Örnek başka bir ilçe
]

HeatMap(heat_data, radius=25).add_to(m)
m
m.save("harita.html")